<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/6_Normalize_tiles_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
"""
STEP: Normalize the 7 input bands across the whole tile dataset.

WHAT THIS DOES (two passes):

PASS 1 - Compute stats:
  Scans every input tile (.npy files from the tiling step) and computes the
  mean and standard deviation for each of the 7 bands (NDVI, NDBI, NDWI,
  Albedo, Elevation, Slope, Aspect), across ALL tiles together — not per
  tile, not per scene. This keeps a given band's scale consistent whether
  you're looking at a 2017 tile or a 2025 tile.

PASS 2 - Apply normalization:
  For every tile, applies z-score normalization to each band:
      normalized_value = (value - band_mean) / band_std
  This puts every band roughly on a similar scale (centered at 0, spread
  of about -3 to +3 for most values), so no single band dominates just
  because its raw numbers happen to be bigger (e.g. Elevation vs NDVI).

  NaN pixels (nodata, e.g. near the AOI boundary within a kept tile) are
  ignored when computing stats, and are left as NaN after normalization —
  they'll be excluded from the loss during training using the mask's own
  nodata marker, so leaving them as NaN here is fine and expected.

WHERE IT READS/WRITES:
  Reads:  TILES/inputs/*.npy  (from the tiling script)
  Writes: TILES/inputs_normalized/*.npy  (new folder, originals untouched)
  Also saves: TILES/band_stats.csv  (mean/std per band, for your records
              and in case you need to normalize new data the same way later)

BEFORE YOU RUN:
- Confirm TILES_FOLDER matches your tiling script's OUTPUT_FOLDER.
- This can take a while with ~4000 tiles — it prints progress every 500 tiles.
"""

import os
import glob
import csv
import numpy as np

TILES_FOLDER = "/content/drive/MyDrive/DATASET/TILES"
INPUTS_FOLDER = os.path.join(TILES_FOLDER, "inputs")
NORMALIZED_FOLDER = os.path.join(TILES_FOLDER, "inputs_normalized")
STATS_CSV = os.path.join(TILES_FOLDER, "band_stats.csv")

BAND_NAMES = ["NDVI", "NDBI", "NDWI", "Albedo", "Elevation", "Slope", "Aspect"]


def compute_band_stats(tile_paths):
    """Pass 1: compute mean/std per band across ALL tiles."""
    num_bands = len(BAND_NAMES)
    # running sums, so we don't have to hold every tile in memory at once
    sums = np.zeros(num_bands, dtype=np.float64)
    sums_sq = np.zeros(num_bands, dtype=np.float64)
    counts = np.zeros(num_bands, dtype=np.float64)

    for i, path in enumerate(tile_paths):
        tile = np.load(path).astype(np.float64)  # shape: (7, H, W)
        for b in range(num_bands):
            band = tile[b]
            valid = band[~np.isnan(band)]
            sums[b] += valid.sum()
            sums_sq[b] += (valid ** 2).sum()
            counts[b] += valid.size

        if (i + 1) % 500 == 0:
            print(f"  Stats pass: processed {i + 1}/{len(tile_paths)} tiles")

    means = sums / counts
    # variance = E[X^2] - (E[X])^2
    variances = (sums_sq / counts) - (means ** 2)
    stds = np.sqrt(np.maximum(variances, 0))  # guard against tiny negative from float error

    return means, stds


def apply_normalization(tile_paths, means, stds):
    """Pass 2: apply z-score normalization and save each tile."""
    os.makedirs(NORMALIZED_FOLDER, exist_ok=True)

    for i, path in enumerate(tile_paths):
        tile = np.load(path).astype(np.float64)
        normalized = np.empty_like(tile)

        for b in range(len(BAND_NAMES)):
            std = stds[b] if stds[b] != 0 else 1.0  # avoid divide-by-zero
            normalized[b] = (tile[b] - means[b]) / std

        tile_id = os.path.basename(path)
        np.save(os.path.join(NORMALIZED_FOLDER, tile_id), normalized.astype(np.float32))

        if (i + 1) % 500 == 0:
            print(f"  Apply pass: processed {i + 1}/{len(tile_paths)} tiles")


def main():
    tile_paths = sorted(glob.glob(os.path.join(INPUTS_FOLDER, "*.npy")))
    if not tile_paths:
        print(f"No tiles found in {INPUTS_FOLDER} — check the path and re-run tiling if needed.")
        return

    print(f"Found {len(tile_paths)} tiles.\n")

    print("PASS 1: computing per-band statistics across all tiles...")
    means, stds = compute_band_stats(tile_paths)

    print("\nBand statistics:")
    with open(STATS_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["band", "mean", "std"])
        for name, mean, std in zip(BAND_NAMES, means, stds):
            print(f"  {name}: mean={mean:.4f}, std={std:.4f}")
            writer.writerow([name, mean, std])

    print(f"\nStats saved to {STATS_CSV}")

    print("\nPASS 2: applying normalization and saving tiles...")
    apply_normalization(tile_paths, means, stds)

    print(f"\nDone. Normalized tiles saved to {NORMALIZED_FOLDER}")


if __name__ == "__main__":
    main()

Found 3942 tiles.

PASS 1: computing per-band statistics across all tiles...
  Stats pass: processed 500/3942 tiles
  Stats pass: processed 1000/3942 tiles
  Stats pass: processed 1500/3942 tiles
  Stats pass: processed 2000/3942 tiles
  Stats pass: processed 2500/3942 tiles
  Stats pass: processed 3000/3942 tiles
  Stats pass: processed 3500/3942 tiles

Band statistics:
  NDVI: mean=0.3696, std=0.1955
  NDBI: mean=0.0220, std=0.1414
  NDWI: mean=-0.4194, std=0.1591
  Albedo: mean=0.1402, std=0.0176
  Elevation: mean=323.9325, std=90.9044
  Slope: mean=2.8192, std=2.8321
  Aspect: mean=151.2472, std=86.7526

Stats saved to /content/drive/MyDrive/DATASET/TILES/band_stats.csv

PASS 2: applying normalization and saving tiles...
  Apply pass: processed 500/3942 tiles
  Apply pass: processed 1000/3942 tiles
  Apply pass: processed 1500/3942 tiles
  Apply pass: processed 2000/3942 tiles
  Apply pass: processed 2500/3942 tiles
  Apply pass: processed 3000/3942 tiles
  Apply pass: processed 35